In [ ]:
# %pip install firthmodels

In [ ]:
# pip install -U kaleido

In [ ]:
# pip install --upgrade kaleido

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import openchord as ocd
# https://github.com/pke1029/open-chord/tree/main
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from firthmodels import FirthLogisticRegression
from firthmodels.adapters.statsmodels import FirthLogit

import plotly.graph_objects as go
import plotly.express as px
import math
import kaleido

import matplotlib.pyplot as plt
from matplotlib import font_manager

import json, textwrap
from IPython.display import IFrame, display

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

# Loading and join

In [ ]:
# Load nvf lookup
csew_vf = pd.read_csv(
    f_root / "data/csew/merged/vf_lookup_TEST.csv",
    dtype=str,
    index_col="global_person_id",
    na_values=["<NA>", "NaN"]
)

In [ ]:
# Load nvf lookup
csew_nvf = pd.read_csv(
    f_root / "data/csew/merged/nvf_lookup.csv",
    dtype=str,
    index_col="global_person_id",
    na_values=["<NA>", "NaN"]
)

In [ ]:
csew_vf_nvf = (
    csew_nvf
    .assign(received_vf=lambda x: x.index.isin(csew_vf.index))
    .join(
        csew_vf,
        how="left",
        rsuffix="_vf",
        validate="one_to_many"
    )
)

csew_vf_nvf["received_vf"] = (csew_vf_nvf["received_vf"].astype("boolean"))
csew_vf_nvf.drop(columns=["Unnamed: 0", "Unnamed: 0_vf", "year", "wave_vf"],inplace=True,errors="ignore")

In [ ]:
vawg_cols = ["vawg_official_codes", "vawg_sex_force_threats","vawg_domestic", "vawg_harasm"]
csew_vf_nvf["vawg"] = (
    csew_vf_nvf[vawg_cols]
    .eq("True")
    .any(axis=1)
    .astype("int")
)

In [ ]:
# csew_vf_nvf.info()

In [ ]:
csew_vf_nvf["sex"].value_counts()

In [ ]:
# Copy female records
df = csew_vf_nvf.loc[csew_vf_nvf["sex"].eq("2")].copy()

# Standardise mixed boolean-like values
bool_map = {
    "1": True, "1.0": True, "true": True, "yes": True,
    "0": False, "0.0": False, "false": False, "no": False
}

df["reported_discard"] = (
    df["reported_discard"].astype("string").str.strip().str.lower()
    .map(bool_map).astype("boolean")
)

df["copsknow_new"] = df["copsknow_new"].astype("string").str.strip()
df["resp_told_police"] = df["resp_told_police"].astype("string").str.strip()

# Incident-level police-known indicator as 0/1
df["police_known_final"] = df["copsknow_new"].eq("1.0")
df.loc[df["resp_told_police"].ne("1.0"), "police_known_final"] = False
df.loc[df["reported_discard"].eq(True), "police_known_final"] = True
df["police_known_final"] = df["police_known_final"].astype("Int64")

# Indicators collapsed using any positive incident
any_true_cols = [
    "vawg", "vawg_official_codes", "vawg_sex_force_threats",
    "vawg_domestic", "vawg_harasm", "reported_discard",
    "police_known_final"
]

# Standardise boolean incident indicators, excluding the integer police column
for col in (set(any_true_cols) & set(df.columns)) - {"police_known_final"}:
    df[col] = (
        df[col].astype("string").str.strip().str.lower()
        .map(bool_map).astype("boolean")
    )

# Complete transformed incident-level data
vawg_incident_level = df.copy()

drop_cols = [
    "pincid", "emotreac", "howcopk", "crime", "offence", "respinj",
    "ofsex", "att", "offrel", "copsknow_new", "resp_told_police",
    *[f"ycopno_{x}" for x in "abcdefghijklmnopqrstuvwx"]
]

# First person-level value; maximum gives 1 if any incident is 1
agg = {
    col: ("max" if col in any_true_cols else "first")
    for col in vawg_incident_level.columns
}

vawg_person_level = (
    vawg_incident_level
    .groupby(level=0, sort=False)
    .agg(agg)
    .rename(columns={"police_known_final": "police_known_all"})
    .drop(columns=drop_cols, errors="ignore")
    .loc[lambda x: x["vawg"].eq(True)]
    .copy()
)

vawg_person_level

In [ ]:
vawg_person_level.info()

In [ ]:
vawg_person_level["gor"].value_counts()

In [ ]:
vawg_person_level = (vawg_person_level.dropna(subset="police_known_all").copy())

In [ ]:
cols = ["agelong","relig3","ethgrp2a","illharmONS2","rnssec3","livharm1a","nslivarr","remploya","polatt7","educint", "nchil2","income_group","managhh2","rural3","inner","imd","imd_crime"]

bar_labels = {
    "agelong": "Age",
    "relig3": "Religion",
    "ethgrp2a": "Ethnicity",
    "illharmONS2": "Disability",
    "rnssec3": "Socioeconomic class",
    # "livharm1a": "Partnered",
    "nslivarr": "Live as couple",
    "remploya": "Employed",
    # "remploy2a": "Partner employment",
    "educint": "Has qualifications",
    "nchil2": "Has children",
    "income_group": "Income",
    "managhh2": "Hard to cover unexpected expenses?",
    "polatt7": "Confident in police",
    "rural3": "Rural area",
    "inner": "Inner city",
    "imd": "Deprived",
    "imd_crime": "IMD Crime"
}

category_labels = {
    "__all__": {
        "<NA>": "Missing",
        "Other": "Other"
    },

    "income_group": {
        "High incomes": "High",
        "Middle incomes": "Middle",
        "Lowest incomes": "Low"
    },

    "managhh2": {
        "Unexpected expenses - no problem": "Easy",
        "Unexpected expenses - problem to find": "Hard",
        "Unexpected expenses -  impossible to find": "Very hard"
    },

    "imd": {
        "Least deprived decile": "Least",
        "Middle deprivation deciles": "Middle",
        "Most deprived decile": "Most"
    },

    "imd_crime": {
        "Least deprived decile": "Least",
        "Middle deprivation deciles": "Middle",
        "Most deprived decile": "Most"
    },

    "educint": {
        "Have qualifications": "Yes",
        "No qualifications": "No"
    },

    "nchil2": {
        "No children in household": "No",
        "Children in household": "Yes"
    },

    "rnssec3": {
        "AB": "Higher",
        "C1": "Interm.",
        "C2_DE": "Routine"
    },

    "polatt7": {
        "Confident in police": "Yes",
        "Not confident in police": "No"
    },

    "illharmONS2": {
        "Not_disabled": "No",
        "Disabled": "Yes"
    },

    "relig3": {
        "Christian": "Christian",
        "No_religion": "No relig.",
        "Non_christian_religion": "Other"
    },

    "remploya": {
        "Employed": "Yes",
        "Unemployed or economically inactive": "No"
    },

    "remploy2a": {
        "Employed": "Yes",
        "Economically inactive": "Inactive",
        "Unemployed": "No"
    },

    # "livharm1a": {
    #     "Married or cohabiting": "Yes",
    #     "Single, divorced or widowed": "No"
    # },

    "nslivarr": {
        "live_as_couple": "Yes",
        "not_live_as_couple": "No"
    },

    "ethgrp2a": {
        "White": "White",
        "Not white": "Minority"
    },

    "inner": {
        "Not inner city": "No",
        "Inner city": "Yes"
    },

    "rural3": {
        "Urban": "No",
        "Rural": "Yes"
    },

    "agelong": {
        "age_16_24": "16–24",
        "age_25_34": "25–34",
        "age_35_64": "35–64",
        "age_65_plus": "65+"
    },
}

plot_df = vawg_person_level.loc[
    vawg_person_level["sex"].astype("string").str.strip().eq("2"),
    cols
].copy()

wrap = lambda x, n=16: "<br>".join(textwrap.wrap(str(x), n))
rename = lambda c, x: category_labels.get(c, {}).get(
    str(x), category_labels.get("__all__", {}).get(str(x), str(x))
)

max_categories = 15
min_label_frequency = 20
palette = px.colors.qualitative.Pastel2

records = []

for col in cols:
    counts = plot_df[col].astype("string").fillna("<NA>").value_counts()
    na = counts.get("<NA>", 0)
    counts = counts.drop("<NA>", errors="ignore")

    if len(counts) > max_categories - 1:
        counts = pd.concat([
            counts.head(max_categories - 2),
            pd.Series({"Other": counts.iloc[max_categories - 2:].sum()})
        ])

    records += [
        {"variable": col, "category": cat, "frequency": n, "rank": rank}
        for rank, (cat, n) in enumerate(counts.items())
    ]

    if na:
        records.append({
            "variable": col,
            "category": "<NA>",
            "frequency": na,
            "rank": 999
        })

d = pd.DataFrame(records)
fig = go.Figure()
x_labels = [wrap(bar_labels.get(c, c)) for c in cols]

for rank in sorted(d.loc[d["rank"] != 999, "rank"].unique()):
    z = d[d["rank"].eq(rank)].set_index("variable")
    y = [z.at[c, "frequency"] if c in z.index else 0 for c in cols]
    cats = [rename(c, z.at[c, "category"]) if c in z.index else "" for c in cols]

    fig.add_bar(
        x=x_labels,
        y=y,
        marker_color=palette[rank % len(palette)],
        marker_line_color="white",
        marker_line_width=.5,
        text=[wrap(cat) if n >= min_label_frequency else "" for cat, n in zip(cats, y)],
        textposition="inside",
        textangle=0,
        customdata=list(zip(cats, y)),
        hovertemplate="%{customdata[0]}<br>Frequency: %{customdata[1]:,}<extra></extra>",
        showlegend=False
    )

z = d[d["rank"].eq(999)].set_index("variable")
y = [z.at[c, "frequency"] if c in z.index else 0 for c in cols]

fig.add_bar(
    x=x_labels,
    y=y,
    marker_color="lightgrey",
    marker_line_color="white",
    marker_line_width=.5,
    text=["Missing" if n >= min_label_frequency else "" for n in y],
    textposition="inside",
    textangle=0,
    hovertemplate="Missing<br>Frequency: %{y:,}<extra></extra>",
    showlegend=False
)

fig.update_layout(
    barmode="stack",
    title="Category frequencies VAWG survivors",
    width=2000,
    height=1400,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(b=150),
    uniformtext=dict(minsize=20, mode="hide"),
    font=dict(family="Roboto Slab")
)

fig.update_xaxes(
    tickangle=0,
    showline=True,
    mirror=True,
    linecolor="lightgrey",
    automargin=True,
    tickfont=dict(family="Roboto Slab Bold",size=20),
)

fig.update_yaxes(
    title="Frequency",
    title_font=dict(family="Roboto Slab",size=20),
    showline=True,
    mirror=True,
    linecolor="lightgrey",
    gridcolor="lightgrey",
    zeroline=False,
    tickfont=dict(family="Roboto Slab",size=20),
)

fig.show()

In [ ]:
# fig.write_image(
#     f_root / "figures/category_frequencies.png",
#     width=2400,
#     height=1200,
#     scale=3
# )

In [ ]:
len(vawg_person_level)

In [ ]:
vawg_person_level.info()

In [ ]:
fillna_cols = [
    "vawg_official_codes",
    "vawg_sex_force_threats",
    "vawg_domestic",
    "vawg_harasm",
]

vawg_person_level[fillna_cols] = vawg_person_level[fillna_cols].fillna(False).astype("boolean")

In [ ]:
model_covariates_inlabru_candidates = [
  "vawg_official_codes", "vawg_sex_force_threats", "vawg_domestic",
  "vawg_harasm", 
    "relig3", "rnssec3", "ethgrp2a",
  "polatt7", "educint", "nchil2", "illharmONS2",
  "nslivarr", "rural3", "inner", 
    "imd", "imd_crime",
]

In [ ]:
model_covariates_inlabru = [
  "vawg_official_codes", "vawg_sex_force_threats", "vawg_domestic",
  "vawg_harasm", 
    "relig3", "rnssec3", "ethgrp2a",
  "polatt7", "educint", "nchil2", "illharmONS2",
  "nslivarr", "rural3", "inner", 
    "imd", "imd_crime", "onspsuid_merged"
]

In [ ]:
from matplotlib.ticker import PercentFormatter

df = vawg_person_level
cols_to_plot = model_covariates_inlabru_candidates
missing = df[cols_to_plot].isna().sum()
non_missing = df[cols_to_plot].notna().sum()
total = missing + non_missing

plot_df = pd.DataFrame({
    "missing_n": missing,
    "non_missing_n": non_missing,
    "missing_pct": missing / total,
    "non_missing_pct": non_missing / total,
})

x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(max(8, len(plot_df) * 1.2), 6))

ax.bar(x, plot_df["non_missing_pct"], label="Non-missing")
ax.bar(
    x,
    plot_df["missing_pct"],
    bottom=plot_df["non_missing_pct"],
    label="Missing"
)

for i, row in enumerate(plot_df.itertuples()):
    if row.non_missing_pct > .03:
        ax.text(
            i, row.non_missing_pct / 2,
            f"Non-null\nn={row.non_missing_n:,}",
            ha="center", va="center", fontsize=8
        )

    if row.missing_pct > .03:
        ax.text(
            i, row.non_missing_pct + row.missing_pct / 2,
            f"NA\nn={row.missing_n:,}",
            ha="center", va="center", fontsize=8
        )
    elif row.missing_n > 0:
        ax.text(
            i, 1.01,
            f"NA n={row.missing_n:,}",
            ha="center", va="bottom", fontsize=8
        )

ax.set(
    xticks=x,
    xticklabels=plot_df.index,
    ylim=(0, 1.08),
    xlabel="Variable",
    ylabel="Share of observations",
    title="Missing and non-missing values by variable"
)

ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Value status")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
fig.savefig(
    "missing_values_by_variable.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
vawg_person_level["gor"].value_counts()

In [ ]:
df = vawg_person_level.copy()
cols_to_plot = model_covariates_inlabru_candidates

# Check required columns
required_cols = ["gor", *cols_to_plot]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

# Long-format missingness summary by GOR and variable
plot_df = (
    df.groupby("gor", dropna=False)[cols_to_plot]
      .agg(["size", "count"])
)

plot_df.columns = [
    f"{variable}_{stat}"
    for variable, stat in plot_df.columns
]

records = []

for gor_value, row in plot_df.iterrows():
    for variable in cols_to_plot:
        total_n = row[f"{variable}_size"]
        non_missing_n = row[f"{variable}_count"]
        missing_n = total_n - non_missing_n

        records.append({
            "gor": gor_value,
            "variable": variable,
            "total_n": total_n,
            "non_missing_n": non_missing_n,
            "missing_n": missing_n,
            "non_missing_pct": non_missing_n / total_n if total_n else np.nan,
            "missing_pct": missing_n / total_n if total_n else np.nan,
        })

plot_df = pd.DataFrame(records)

# Preserve variable order
plot_df["variable"] = pd.Categorical(
    plot_df["variable"],
    categories=cols_to_plot,
    ordered=True
)

gor_values = plot_df["gor"].drop_duplicates().tolist()

n_cols = 3
n_rows = int(np.ceil(len(gor_values) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(
        max(14, len(cols_to_plot) * 1.1),
        n_rows * 5
    ),
    sharey=True
)

axes = np.atleast_1d(axes).ravel()
x = np.arange(len(cols_to_plot))

for ax, gor_value in zip(axes, gor_values):
    subset = (
        plot_df.loc[plot_df["gor"] == gor_value]
        .sort_values("variable")
        .set_index("variable")
        .reindex(cols_to_plot)
        .reset_index()
    )

    ax.bar(
        x,
        subset["non_missing_pct"],
        label="Non-missing"
    )

    ax.bar(
        x,
        subset["missing_pct"],
        bottom=subset["non_missing_pct"],
        label="Missing"
    )

    for i, row in enumerate(subset.itertuples()):
        if row.non_missing_pct > 0.08:
            ax.text(
                i,
                row.non_missing_pct / 2,
                f"n={row.non_missing_n:,}",
                ha="center",
                va="center",
                fontsize=7
            )

        if row.missing_pct > 0.08:
            ax.text(
                i,
                row.non_missing_pct + row.missing_pct / 2,
                f"NA\nn={row.missing_n:,}",
                ha="center",
                va="center",
                fontsize=7
            )
        elif row.missing_n > 0:
            ax.text(
                i,
                1.01,
                f"NA {row.missing_n:,}",
                ha="center",
                va="bottom",
                fontsize=6,
                rotation=90
            )

    ax.set_title(str(gor_value))
    ax.set_xticks(x)
    ax.set_xticklabels(
        cols_to_plot,
        rotation=45,
        ha="right"
    )
    ax.set_ylim(0, 1.10)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.spines[["top", "right"]].set_visible(False)

# Remove unused panels
for ax in axes[len(gor_values):]:
    ax.remove()

fig.suptitle(
    "Missing and non-missing values by variable and GOR",
    y=1.02
)

fig.supxlabel("Variable")
fig.supylabel("Share of observations")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Value status",
    loc="upper right"
)

plt.tight_layout()

fig.savefig(
    "missing_values_by_variable_and_gor.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
dropna_cols_mrp = ["agelong", "remploya", "police_known_all"]

vawg_person_level = (
    vawg_person_level
    .dropna(subset=dropna_cols_mrp)
    .copy()
)
len(vawg_person_level)

vawg_person_level_light = (
    vawg_person_level
    .dropna(subset=model_covariates_inlabru)
    .copy()
)
len(vawg_person_level_light)

In [ ]:
df = vawg_person_level_light
cols_to_plot = model_covariates_inlabru
missing = df[cols_to_plot].isna().sum()
non_missing = df[cols_to_plot].notna().sum()
total = missing + non_missing

plot_df = pd.DataFrame({
    "missing_n": missing,
    "non_missing_n": non_missing,
    "missing_pct": missing / total,
    "non_missing_pct": non_missing / total,
})

x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(max(8, len(plot_df) * 1.2), 6))

ax.bar(x, plot_df["non_missing_pct"], label="Non-missing")
ax.bar(
    x,
    plot_df["missing_pct"],
    bottom=plot_df["non_missing_pct"],
    label="Missing"
)

for i, row in enumerate(plot_df.itertuples()):
    if row.non_missing_pct > .03:
        ax.text(
            i, row.non_missing_pct / 2,
            f"Non-null\nn={row.non_missing_n:,}",
            ha="center", va="center", fontsize=8
        )

    if row.missing_pct > .03:
        ax.text(
            i, row.non_missing_pct + row.missing_pct / 2,
            f"NA\nn={row.missing_n:,}",
            ha="center", va="center", fontsize=8
        )
    elif row.missing_n > 0:
        ax.text(
            i, 1.01,
            f"NA n={row.missing_n:,}",
            ha="center", va="bottom", fontsize=8
        )

ax.set(
    xticks=x,
    xticklabels=plot_df.index,
    ylim=(0, 1.08),
    xlabel="Variable",
    ylabel="Share of observations",
    title="Missing and non-missing values by variable"
)

ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Value status")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
fig.savefig(
    "missing_values_by_variable.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
vawg_person_level_light.info()

In [ ]:
vawg_person_level_light["educint"].value_counts()

In [ ]:
vawg_person_level_light["gor"].value_counts()

In [ ]:
vawg_person_level_light.to_csv(f_root / "data/csew/exp_for_inla_stage/export_for_inla_2.csv")

# Testing the covariate set

In [ ]:
reference_categories = {
    "income_group": "Middle incomes",
    "managhh2": "Unexpected expenses - problem to find",
    "agelong": "age_16_24",
    "relig3": "no_religion",
    "ethgrp2a": "White",
    "imd": "Middle deprivation deciles",
    "imd_crime": "Middle deprivation deciles",
    "educint": "no_qualifications",
    "nchil2": "No children in household",
    "rnssec3": "C1",
    "rnssec5": "Never worked or unemployed",
    "polatt7": "Confident in police",
    "illharmONS2": "not_disabled",
    "remploya": "unemployed_or_economically_inactive",
    "nslivarr": "not_live_as_couple",
    "inner": "Not inner city",
    "rural3": "Rural",
    "indlon": "London",
    "gor": "London",
    "oa_sup11": "Hard-pressed living",  
    # "pincid": "single incident",
    # "emotreac": "Not emotional after",
    # "crime": "Something that happens",
    # "respinj": "Was not injured",
    # "att": "No medical attention",
    # "offrel": "other/stranger",

    "vawg_official_codes": False,
    "vawg_sex_force_threats": False,
    "vawg_domestic": False,
    "vawg_harasm": False

}

# GLM Binary Logistic Regression

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, statsmodels.api as sm
from scipy.special import logit
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, balanced_accuracy_score,
                             roc_auc_score, average_precision_score, log_loss,
                             brier_score_loss)

In [ ]:
RANDOM_STATE, outcome = 123, "police_known_all"

model_sets = {
    "age_disability": ["agelong", "illharmONS2"],
    # "religion_disability": ["relig3", "illharmONS2"],
    "age_economic_inactivity": ["agelong", "remploya"],
    "age_ethnicity": ["agelong", "ethgrp2a"],
    "age_living_arrangements": ["agelong", "nslivarr"],
}

# Common complete-case sample and common split
covs = list(dict.fromkeys(v for vs in model_sets.values() for v in vs))
df = vawg_person_level[[outcome] + covs].copy()
df[outcome] = pd.to_numeric(df[outcome], errors="coerce")

for c in covs:
    df[c] = df[c].astype("object")
    if c in reference_categories:
        ref, obs = reference_categories[c], df[c].dropna().unique()
        if ref not in obs:
            raise ValueError(f"{ref!r} absent from {c!r}")
        df[c] = pd.Categorical(df[c], categories=[ref] + [x for x in obs if x != ref])

df = df.dropna().copy()
df[outcome] = df[outcome].astype(int)

train_i, test_i = train_test_split(
    np.arange(len(df)), test_size=.20, stratify=df[outcome],
    random_state=RANDOM_STATE
)

results, fitted_models, influential_cases = [], {}, {}
fig, axes = plt.subplots(
    int(np.ceil(len(model_sets) / 2)), 2,
    figsize=(13, 3.2 * int(np.ceil(len(model_sets) / 2)))
)
axes = np.asarray(axes).ravel()

for ax, (name, vars_) in zip(axes, model_sets.items()):

    X = sm.add_constant(
        pd.get_dummies(df[vars_], drop_first=True, dtype=float),
        has_constant="add"
    )
    y = df[outcome]
    Xtr, Xte, ytr, yte = X.iloc[train_i], X.iloc[test_i], y.iloc[train_i], y.iloc[test_i]

    # Initial model and Cook's distance
    initial = sm.GLM(ytr, Xtr, family=sm.families.Binomial()).fit(maxiter=2000)
    cooks = initial.get_influence(observed=False).cooks_distance[0]
    threshold, cut = 4 / len(Xtr), cooks > 4 / len(Xtr)

    influential_cases[name] = (
        df.iloc[train_i].iloc[np.flatnonzero(cut)]
        .assign(cooks_distance=cooks[cut])
    )

    # Colour = outcome; black outline = above Cook's threshold
    for value, label, colour in [(0, "Unreported", "tab:blue"),
                                 (1, "Reported", "tab:orange")]:
        m = ytr.to_numpy() == value
        ax.scatter(np.flatnonzero(m), cooks[m], s=13, color=colour,
                   alpha=.65, label=label)

    # ax.scatter(np.flatnonzero(cut), cooks[cut], s=38, facecolors="none",
    #            edgecolors="black", linewidths=1, label="Above Cook's threshold")
    ax.axhline(threshold, color="black", linestyle="--",
               label=f"4/n = {threshold:.4f}")
    ax.set(title=f"{name}\n{cut.sum()} above threshold",
           xlabel="Training observation", ylabel="Cook's distance")
    ax.legend(fontsize=7)

    # Remove influential training cases and refit
    keep = np.flatnonzero(~cut)
    model = sm.GLM(
        ytr.iloc[keep], Xtr.iloc[keep],
        family=sm.families.Binomial()
    ).fit(maxiter=2000)

    fitted_models[name] = model

    print("\n" + "=" * 85)
    print(f"{name}: {' + '.join(vars_)} | removed {cut.sum()}/{len(Xtr)}")
    print(model.summary())

    # Test performance at threshold 0.50
    prob = np.clip(model.predict(Xte), 1e-8, 1 - 1e-8)
    pred = (prob >= .50).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan

    cal = sm.GLM(
        yte.to_numpy(),
        sm.add_constant(logit(prob)),
        family=sm.families.Binomial()
    ).fit()

    results.append({
        # Model
        "Predictor set": " + ".join(vars_),
        "ROC AUC": roc_auc_score(yte, prob),
    
        # Classification performance
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        # "Positive predictive value": ppv,
        # "Negative predictive value": npv,
        "Accuracy": accuracy_score(yte, pred),
        "Balanced accuracy": balanced_accuracy_score(yte, pred),
    
        # Classification counts
        "True positives": tp,
        "False negatives": fn,
        "True negatives": tn,
        "False positives": fp,
    
        # Model and sample diagnostics
        # "Test sample size": len(yte),
        "Influential observations removed": int(cut.sum()),
        "Final training sample size": len(keep),
        
        # "PR AUC": average_precision_score(yte, prob),
        # "Brier score": brier_score_loss(yte, prob),
        # "Log loss": log_loss(yte, prob),
        # "Calibration intercept": cal.params.iloc[0],
        # "Calibration slope": cal.params.iloc[1],
        # "AIC": model.aic,
        # "BIC": model.bic_llf,
    })

for ax in axes[len(model_sets):]:
    ax.set_visible(False)

fig.suptitle("Cook's distance: colour indicates reporting outcome", y=1.01)
fig.tight_layout()
plt.show()

comparison_table = (
    pd.DataFrame(results)
    .sort_values(
        ["ROC AUC", ],
        ascending=False
    )
    .reset_index(drop=True)
)

metric_cols = ["Sensitivity","Specificity",
               #"Positive predictive value","Negative predictive value",
               "Accuracy","Balanced accuracy", "ROC AUC"]
comparison_table[metric_cols] = comparison_table[metric_cols].round(3)
display(comparison_table)

In [ ]:
comparison_table.to_excel(
    f_root / "figures/comparison_table_glm.xlsx",
    index=False
)

In [ ]:
num_official_codes_offences = len(vawg_incident_level[vawg_incident_level["vawg_official_codes"]== True])

vawg_cols = ["vawg_official_codes", "vawg_sex_force_threats","vawg_domestic", "vawg_harasm"]
vawg_incident_level_calc = vawg_incident_level.copy()
vawg_incident_level_calc["vawg"] = (
    vawg_incident_level_calc[vawg_cols]
    .fillna(False)
    .astype("boolean")
    .any(axis=1)
)
num_vawg_offences = len(vawg_incident_level[vawg_incident_level["vawg"]== True])

print(
    f"The inclusion of the proposed filters increased the sample size from {num_official_codes_offences} (codes only) to {num_vawg_offences} incidents of VAWG."
)


In [ ]:
# EXPORT FOR venn diagram that is produced in R

vawg_incident_level.to_csv(
    f_root / "data/csew/vawg_incident_level.csv",
    index=True
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.transforms import ScaledTranslation
import eunoia as eu

vawg_cols = [
    "vawg_official_codes",
    "vawg_sex_force_threats",
    "vawg_domestic",
    "vawg_harasm"
]

names = ["Official", "Force/threats", "Domestic", "Harassment"]
base_colors = ["#3B82A0", "#D46A4C", "#4F9A68", "#8E6BB8"]

plt.rcParams["font.family"] = "Roboto Slab"

data = (
    vawg_incident_level[vawg_cols]
    .fillna(False)
    .astype(bool)
    .reset_index(drop=True)
)

incident_ids = data.index.to_numpy()
sets = {
    name: set(incident_ids[data[col].to_numpy()])
    for name, col in zip(names, vawg_cols)
}
totals = {name: len(sets[name]) for name in names}

fit = eu.euler(sets, shape="ellipse")
ax = fit.plot(
    quantities={
        "type": "counts",
        "fontsize": 8
    },
    edges={"linewidth": 0.1}
)

fig = ax.figure
fig.set_size_inches(9, 12)
ax.set_axis_off()

# Force colours onto the actual diagram patches
for patch, col in zip(ax.patches[:4], base_colors):
    patch.set_facecolor(to_rgba(col, 0.55))
    # patch.set_edgecolor("none")

label_offsets = {
    "Official": (0, -24),
    "Force/threats": (0, -24),
    "Domestic": (0, 14),
    "Harassment": (0, 14)
}

for txt in ax.texts:
    label = txt.get_text().strip()

    if label in totals:
        txt.set_text(f"{label}\n(total n={totals[label]:,})")
        txt.set_fontsize(11)
        txt.set_fontweight("bold")
        dx, dy = label_offsets.get(label, (0, 0))
        txt.set_transform(
            txt.get_transform() +
            ScaledTranslation(dx / 72, dy / 72, fig.dpi_scale_trans)
        )
    else:
        try:
            txt.set_text(f"{float(label):,.0f}")
        except ValueError:
            pass

fig.savefig(
    f_root / "figures/euler_vawg_definitions.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
fig.savefig(
    f_root / "figures/euler_vawg_definitions.pdf",
    bbox_inches="tight",
    facecolor="white"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import eunoia as eu

plt.rcParams["font.family"] = "Roboto Slab"

cols = {
    "Official": "vawg_official_codes",
    "Rape": "whatfo4f",
    "Attempted rape": "whatfo4g",
    "Sexual assault": "whatfo4h",
    "Sexual element": "v712"
}

d = vawg_incident_level[list(cols.values())].copy()

d["Official"] = d["vawg_official_codes"].fillna(False).astype(bool)

for label, var in {
    "Rape": "whatfo4f",
    "Attempted rape": "whatfo4g",
    "Sexual assault": "whatfo4h",
    "Sexual element": "v712"
}.items():
    d[label] = (
        d[var]
        .fillna("")
        .astype(str)
        .str.contains("1", regex=False)
    )

d = d[list(cols.keys())].reset_index(drop=True)
ids = d.index.to_numpy()

sets = {c: set(ids[d[c].to_numpy()]) for c in d.columns}
totals = {c: len(s) for c, s in sets.items()}

fit = eu.euler(sets, shape="ellipse")
ax = fit.plot(
    quantities={"type": "counts", "fontsize": 8},
    edges={"linewidth": 0.1}
)

fig = ax.figure
fig.set_size_inches(10, 10)
ax.set_axis_off()

colors = ["#3B82A0", "#4F9A68", "#8E6BB8", "#D6A84B", "#6E8FA3"]
for patch, col in zip(ax.patches[:5], colors):
    patch.set_facecolor(to_rgba(col, 0.5))

for txt in ax.texts:
    label = txt.get_text().strip()

    if label in totals:
        txt.set_text(f"{label}\n(n={totals[label]:,})")
        txt.set_fontsize(10)
        txt.set_fontweight("bold")
    else:
        try:
            txt.set_text(f"{float(label):,.0f}")
        except ValueError:
            pass

fig.savefig(
    f_root / "figures/euler_vawg_official_sexual_variables.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

In [ ]:
vawg_incident_level

In [ ]:
len(vawg_only)

In [ ]:
# First restrict to VAWG incidents
vawg_only = vawg_incident_level[
    vawg_incident_level["vawg"].fillna(False).isin(
        [True, 1, "1", "TRUE", "True", "true"]
    )
].copy()

# Then identify VAWG incidents not captured by the official flag
official = (
    vawg_only["vawg_official_codes"]
    .fillna(False)
    .isin([True, 1, "1", "TRUE", "True", "true"])
)

non_official = vawg_only.loc[~official]

print("VAWG incidents not captured as Official:", len(non_official))
print("\nOffence codes:")
print(non_official["offence"].value_counts(dropna=False))

In [ ]:
# First restrict to VAWG incidents
vawg_only = vawg_incident_level[
    vawg_incident_level["vawg"].fillna(False).isin(
        [True, 1, "1", "TRUE", "True", "true"]
    )
].copy()

# Identify official and sexual force/threats flags
official = (
    vawg_only["vawg_official_codes"]
    .fillna(False)
    .isin([True, 1, "1", "TRUE", "True", "true"])
)

sexual_threats = (
    vawg_only["vawg_sex_force_threats"]
    .fillna(False)
    .isin([True, 1, "1", "TRUE", "True", "true"])
)

# Keep incidents not captured by official coding
# but captured by the sexual force/threats flag
non_official_sexual = vawg_only.loc[
    (~official) & sexual_threats
]

print(
    "VAWG incidents not captured as Official "
    "but captured by sexual force/threats:",
    len(non_official_sexual)
)

print("\nOffence codes:")
print(non_official_sexual["offence"].value_counts(dropna=False))

In [ ]:
# First restrict to VAWG incidents
vawg_only = vawg_incident_level[
    vawg_incident_level["vawg"].fillna(False).isin(
        [True, 1, "1", "TRUE", "True", "true"]
    )
].copy()

# Define flags
official = (
    vawg_only["vawg_official_codes"]
    .fillna(False)
    .isin([True, 1, "1", "TRUE", "True", "true"])
)

sexual_threats = (
    vawg_only["vawg_sex_force_threats"]
    .fillna(False)
    .isin([True, 1, "1", "TRUE", "True", "true"])
)

v712_flag = (
    vawg_only["v712"]
    .fillna("")
    .astype(str)
    .str.contains("1", regex=False)
)

# Keep incidents:
# - not Official
# - flagged by sexual force/threats
# - not flagged because of v712
non_official_sexual = vawg_only.loc[
    (~official) & sexual_threats & (~v712_flag)
]

print(
    "VAWG incidents not captured as Official, "
    "captured by sexual force/threats, excluding v712:",
    len(non_official_sexual)
)

print("\nOffence codes:")
print(non_official_sexual["offence"].value_counts(dropna=False))

In [ ]:
df = vawg_incident_level.copy()
df = df[df["vawg"]== True]

# Count how many incident rows each person ID has
person_counts = df.index.value_counts()

# Count how many people appear 1 time, 2 times, 3 times, etc.
repeat_distribution = (
    person_counts
    .value_counts()
    .sort_index()
)

# Colour palette
mean_cols = [
    "#543005", "#8c510a", "#bf812d", "#dfc27d", "#f6e8c3",
    "#c7eae5", "#80cdc1", "#35978f", "#01665e", "#003c30"
]

# Use Roboto Slab if installed
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_family = "Roboto Slab" if "Roboto Slab" in available_fonts else "serif"

plt.rcParams["font.family"] = font_family

fig, ax = plt.subplots(figsize=(9, 2))

colors = [
    mean_cols[i % len(mean_cols)]
    for i in range(len(repeat_distribution))
]

bars = ax.bar(
    repeat_distribution.index,
    repeat_distribution.values,
    color=colors,
    width=0.75
)

# Add counts above bars
for bar, n in zip(bars, repeat_distribution.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{n:,}",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_xlabel("Number of incidents per person", fontsize=11)
ax.set_ylabel("Number of people", fontsize=11)

ax.set_title(
    "Number of VAWG incidents recorded per person",
    fontsize=15,
    pad=15
)

ax.set_xticks(repeat_distribution.index)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(
    axis="y",
    alpha=0.15,
    linewidth=0.7
)

plt.tight_layout()

fig.savefig(
    "vawg_incidents_per_person.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

fig.savefig(
    f_root / "figures/vawg_incidents_per_person.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

In [ ]:
vawg_incident_level

In [ ]:
vawg_cols = ["vawg_official_codes","vawg_sex_force_threats","vawg_domestic","vawg_harasm"]
names_inc = ["over 20 incidents in series", "10 to 20 incidents in series","2 to 10 incidents in series","single incident"]

df = vawg_incident_level.copy()
df = df[df["vawg"]== True].copy()
df[vawg_cols] = df[vawg_cols].fillna(False).astype(bool)
df["vawg"] = df[vawg_cols].any(axis=1)

sankey_df = df.loc[df.vawg,["inc_num_uncapped","offrel",*vawg_cols]].copy()
sankey_df["inc_num_uncapped"] = sankey_df["inc_num_uncapped"].fillna("incident number unavailable").astype(str)
sankey_df["offrel"] = sankey_df["offrel"].fillna("offender status question unanswered").astype(str)
sankey_df["identification_route"] = sankey_df.vawg_official_codes.map(
    {True:"Official codes",False:"Other routes"}
)

n_total = len(sankey_df)
inc_levels = names_inc
# inc_levels = [x for x in names_inc if x in sankey_df.inc_num_uncapped.unique()]
# inc_levels += [x for x in sankey_df.inc_num_uncapped.unique() if x not in inc_levels]
offrel_levels = sankey_df.offrel.value_counts().index.tolist()
route_levels = ["Official codes","Other routes"]

inc_counts = sankey_df.inc_num_uncapped.value_counts()
offrel_counts = sankey_df.offrel.value_counts()
route_counts = sankey_df.identification_route.value_counts()

inc_offrel = sankey_df.groupby(["inc_num_uncapped","offrel"],observed=True).size().reset_index(name="value")
offrel_route = sankey_df.groupby(["offrel","identification_route"],observed=True).size().reset_index(name="value")

wrap = lambda x: "\n".join(textwrap.wrap(str(x),30,break_long_words=False,break_on_hyphens=False))

inc_cols = dict(zip(inc_levels,["#a3cef1","#ffd670","#90be6d","#ef233c","#cccccc"][:len(inc_levels)]))
offrel_palette = ["#90DBF4","#A3C4F3","#B9FBC0","#98F5E1","#CFBAF0","#F1C0E8",
                  "#FFCFD2","#FDE4CF","#FBF8CC","#A9DEF9","#CDEAC0","#FFC8DD","#D8E2DC","#E4C1F9"]
offrel_cols = {x:offrel_palette[i%len(offrel_palette)] for i,x in enumerate(offrel_levels)}
route_cols = {"Official codes":"#FF6B8A","Other routes":"#72D6A0"}

def node(name,depth,label,count,colour):
    return {"name":name,"depth":depth,
            "displayLabel":f"{wrap(label)}\n(n={count:,}; {count/n_total:.1%})",
            "itemStyle":{"color":colour,"borderColor":"#FFFFFF","borderWidth":3}}

# nodes = [node("all",0,"All VAWG offences",n_total,"#FFD166")]
nodes = [node(f"inc_{i}",0,x,int(inc_counts[x]),inc_cols[x]) for i,x in enumerate(inc_levels)]
nodes += [node(f"offrel_{i}",1,x,int(offrel_counts[x]),offrel_cols[x]) for i,x in enumerate(offrel_levels)]
nodes += [node(f"route_{i}",2,x,int(route_counts.get(x,0)),route_cols[x]) for i,x in enumerate(route_levels)]

inc_ids = {x:f"inc_{i}" for i,x in enumerate(inc_levels)}
offrel_ids = {x:f"offrel_{i}" for i,x in enumerate(offrel_levels)}
route_ids = {x:f"route_{i}" for i,x in enumerate(route_levels)}

# links = [{"source":"all","target":inc_ids[x],"value":int(inc_counts[x])} for x in inc_levels]
links = [{"source":inc_ids[r.inc_num_uncapped],"target":offrel_ids[r.offrel],"value":int(r.value)}
          for r in inc_offrel.itertuples(index=False)]
links += [{"source":offrel_ids[r.offrel],"target":route_ids[r.identification_route],"value":int(r.value)}
          for r in offrel_route.itertuples(index=False)]

html = f"""
<!DOCTYPE html><html><head>
<meta charset="UTF-8">
<link href="https://fonts.googleapis.com/css2?family=Roboto+Slab:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/echarts@5/dist/echarts.min.js"></script>
<style>
html,body,#chart{{width:100%;height:100%;margin:0;background:white;font-family:"Roboto Slab",serif}}
#chart{{height:950px}}
</style></head><body><div id="chart"></div><script>

const nodes={json.dumps(nodes,ensure_ascii=False)};
const links={json.dumps(links,ensure_ascii=False)};
const chart=echarts.init(document.getElementById("chart"),null,{{renderer:"svg"}});

chart.setOption({{
 backgroundColor:"#fff",
 tooltip:{{
   trigger:"item",
   textStyle:{{fontFamily:"Roboto Slab",fontSize:22}},
   formatter:p=>p.dataType==="edge"
     ? nodes.find(x=>x.name===p.data.source).displayLabel.replaceAll("\\n"," ")+" → "+
       nodes.find(x=>x.name===p.data.target).displayLabel.replaceAll("\\n"," ")+"<br><b>n="+
       Number(p.data.value).toLocaleString()+"</b>"
     : p.data.displayLabel.replaceAll("\\n","<br>")
 }},
 toolbox:{{show:true,right:25,top:20,feature:{{
   saveAsImage:{{name:"vawg_sankey",type:"png",pixelRatio:3,backgroundColor:"#fff"}}
 }}}},
 series:[{{
   type:"sankey",data:nodes,links:links,
   left:"3%",right:"11%",top:65,bottom:30,
   orient:"horizontal",nodeAlign:"justify",nodeWidth:28,nodeGap:18,
   draggable:true,layoutIterations:0,
   emphasis:{{focus:"adjacency"}},
   label:{{
     show:true,position:"right",distance:8,color:"#000",
     fontFamily:"Roboto Slab",fontSize:22,fontWeight:500,lineHeight:20,
     textBorderColor:"#fff",textBorderWidth:5,
     formatter:p=>p.data.displayLabel
   }},
   itemStyle:{{borderColor:"#fff",borderWidth:3}},
   lineStyle:{{color:"gradient",opacity:.58,curveness:.52}}
 }}]
}});

window.addEventListener("resize",()=>chart.resize());
</script></body></html>
"""

output_file = Path("vawg_incnum_offrel_sankey.html")   
output_file.write_text(html,encoding="utf-8")
display(IFrame(src=str(output_file),width="100%",height=980))

In [ ]:
len(df)

In [ ]:
vawg_cols = ["vawg_official_codes","vawg_sex_force_threats","vawg_domestic","vawg_harasm"]
names_inc = ["over 20 incidents in series","10 to 20 incidents in series",
             "2 to 10 incidents in series","single incident"]

true_vals = {True, 1, "1", "TRUE", "True", "true"}
df

df = vawg_incident_level.loc[
    vawg_incident_level["agelong"].astype(str).eq("age_65_plus")
].copy()
df = df[df["vawg"]== True].copy()

for c in vawg_cols:
    df[c] = df[c].apply(lambda x: x in true_vals)

df["vawg"] = df[vawg_cols].any(axis=1)

sankey_df = df.loc[df["vawg"], ["inc_num_uncapped","offrel",*vawg_cols]].copy()

if sankey_df.empty:
    raise ValueError("No VAWG incidents found for age_65_plus.")

sankey_df["inc_num_uncapped"] = sankey_df["inc_num_uncapped"].fillna(
    "incident number unavailable"
).astype(str)

sankey_df["offrel"] = sankey_df["offrel"].fillna(
    "offender status question unanswered"
).astype(str)

sankey_df["identification_route"] = np.where(
    sankey_df["vawg_official_codes"],
    "Official codes",
    "Other routes"
)

n_total = len(sankey_df)

inc_levels = [x for x in names_inc if x in sankey_df["inc_num_uncapped"].unique()]
offrel_levels = sankey_df["offrel"].value_counts().index.tolist()
route_levels = [x for x in ["Official codes","Other routes"]
                if x in sankey_df["identification_route"].unique()]

inc_counts = sankey_df["inc_num_uncapped"].value_counts()
offrel_counts = sankey_df["offrel"].value_counts()
route_counts = sankey_df["identification_route"].value_counts()

inc_offrel = (
    sankey_df.groupby(["inc_num_uncapped","offrel"], observed=True)
    .size().reset_index(name="value")
)

offrel_route = (
    sankey_df.groupby(["offrel","identification_route"], observed=True)
    .size().reset_index(name="value")
)

wrap = lambda x: "\n".join(
    textwrap.wrap(str(x),30,break_long_words=False,break_on_hyphens=False)
)

inc_cols = dict(zip(
    inc_levels,
    ["#a3cef1","#ffd670","#90be6d","#ef233c"][:len(inc_levels)]
))

offrel_palette = [
    "#90DBF4","#A3C4F3","#B9FBC0","#98F5E1","#CFBAF0","#F1C0E8",
    "#FFCFD2","#FDE4CF","#FBF8CC","#A9DEF9","#CDEAC0","#FFC8DD",
    "#D8E2DC","#E4C1F9"
]

offrel_cols = {
    x: offrel_palette[i % len(offrel_palette)]
    for i, x in enumerate(offrel_levels)
}

route_cols = {
    "Official codes":"#FF6B8A",
    "Other routes":"#72D6A0"
}

def node(name, depth, label, count, colour):
    return {
        "name": name,
        "depth": depth,
        "displayLabel": f"{wrap(label)}\n(n={count:,}; {count/n_total:.1%})",
        "itemStyle": {
            "color": colour,
            "borderColor": "#FFFFFF",
            "borderWidth": 3
        }
    }

nodes = (
    [node(f"inc_{i}",0,x,int(inc_counts[x]),inc_cols[x])
     for i,x in enumerate(inc_levels)]
    +
    [node(f"offrel_{i}",1,x,int(offrel_counts[x]),offrel_cols[x])
     for i,x in enumerate(offrel_levels)]
    +
    [node(f"route_{i}",2,x,int(route_counts[x]),route_cols[x])
     for i,x in enumerate(route_levels)]
)

inc_ids = {x:f"inc_{i}" for i,x in enumerate(inc_levels)}
offrel_ids = {x:f"offrel_{i}" for i,x in enumerate(offrel_levels)}
route_ids = {x:f"route_{i}" for i,x in enumerate(route_levels)}

links = [
    {"source":inc_ids[r.inc_num_uncapped],
     "target":offrel_ids[r.offrel],
     "value":int(r.value)}
    for r in inc_offrel.itertuples(index=False)
    if r.inc_num_uncapped in inc_ids
]

links += [
    {"source":offrel_ids[r.offrel],
     "target":route_ids[r.identification_route],
     "value":int(r.value)}
    for r in offrel_route.itertuples(index=False)
]

html = f"""
<!DOCTYPE html><html><head>
<meta charset="UTF-8">
<link href="https://fonts.googleapis.com/css2?family=Roboto+Slab:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/echarts@5/dist/echarts.min.js"></script>
<style>
html,body,#chart{{width:100%;height:100%;margin:0;background:white;font-family:"Roboto Slab",serif}}
#chart{{height:950px}}
</style></head><body><div id="chart"></div><script>

const nodes={json.dumps(nodes,ensure_ascii=False)};
const links={json.dumps(links,ensure_ascii=False)};
const chart=echarts.init(document.getElementById("chart"),null,{{renderer:"svg"}});

chart.setOption({{
 backgroundColor:"#fff",
 tooltip:{{
   trigger:"item",
   textStyle:{{fontFamily:"Roboto Slab",fontSize:22}},
   formatter:p=>p.dataType==="edge"
     ? nodes.find(x=>x.name===p.data.source).displayLabel.replaceAll("\\n"," ")+" → "+
       nodes.find(x=>x.name===p.data.target).displayLabel.replaceAll("\\n"," ")+"<br><b>n="+
       Number(p.data.value).toLocaleString()+"</b>"
     : p.data.displayLabel.replaceAll("\\n","<br>")
 }},
 toolbox:{{show:true,right:25,top:20,feature:{{
   saveAsImage:{{name:"vawg_sankey_age_65_plus",type:"png",pixelRatio:3,backgroundColor:"#fff"}}
 }}}},
 series:[{{
   type:"sankey",data:nodes,links:links,
   left:"3%",right:"11%",top:65,bottom:30,
   orient:"horizontal",nodeAlign:"justify",nodeWidth:28,nodeGap:18,
   draggable:true,layoutIterations:0,
   emphasis:{{focus:"adjacency"}},
   label:{{
     show:true,position:"right",distance:8,color:"#000",
     fontFamily:"Roboto Slab",fontSize:22,fontWeight:500,lineHeight:20,
     textBorderColor:"#fff",textBorderWidth:5,
     formatter:p=>p.data.displayLabel
   }},
   itemStyle:{{borderColor:"#fff",borderWidth:3}},
   lineStyle:{{color:"gradient",opacity:.58,curveness:.52}}
 }}]
}});

window.addEventListener("resize",()=>chart.resize());
</script></body></html>
"""

output_file = Path("vawg_incnum_offrel_sankey_age_65_plus.html")
output_file.write_text(html, encoding="utf-8")

display(IFrame(src=str(output_file), width="100%", height=980))